In [2]:
from collections import defaultdict
import pickle
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.nn import DistMult

from tqdm import tqdm


device = 'cuda' if torch.cuda.is_available() else 'cpu'



In [3]:
with open("../data/dicts/bert_embeddings/protein_esm_35M.pkl", 'rb') as f:
    emb_prot = pickle.load(f)
with open("../data/dicts/bert_embeddings/sm_chemberta_10M_MTR.pkl", 'rb') as f:
    emb_sm = pickle.load(f)
with open("../data/dicts/bert_embeddings/rna_berta.pkl", 'rb') as f:
    emb_rna = pickle.load(f)
with open("../data/dicts/bert_embeddings/dna_full.pkl", 'rb') as f:
    emb_dna = pickle.load(f)

rawid2enb = emb_prot | emb_sm | emb_rna | emb_dna

In [5]:


node_types = ['AA', 'DNA', 'RNA', 'SmallMolecule']
type2id = {name: i for i, name in enumerate(node_types)}

rawid_to_local = {}
rawid_to_type = {}
raw_tensors = {}

for t_name in node_types:
    # Читаем ТОЛЬКО колонку с ID, чтобы узнать тип узлов
    df_nodes = pd.read_csv(f'../data/nodes/nodes_for_mdm/{t_name}.csv', usecols=['id_entity'])

    embeddings_list = []

    for local_idx, raw_id in enumerate(df_nodes['id_entity']):
        # Заполняем маппинги для DataLoader
        rawid_to_local[raw_id] = local_idx
        rawid_to_type[raw_id] = type2id[t_name]

        # Достаем готовый эмбеддинг из твоего словаря
        emb = rawid2enb[raw_id]

        embeddings_list.append(emb)

    # Склеиваем список тензоров в одну матрицу для этого типа
    # Получится тензор размерности [N_nodes_of_this_type, embedding_dim]
    raw_tensors[str(type2id[t_name])] = torch.stack(embeddings_list)

df_edges = pd.read_csv('../data/edges/clean_edges_without_NaNm.csv')

# Превращаем предикаты в числа (как у тебя)
pred_keys, pred_values = pd.factorize(df_edges['predicate'])
df_edges['predicate'] = pred_keys

# ГЛАВНАЯ МАГИЯ: Мапим сырые ID сразу в локальные индексы и типы
h_local = df_edges['id_entity_1'].map(rawid_to_local).to_numpy()
h_type  = df_edges['id_entity_1'].map(rawid_to_type).to_numpy()

t_local = df_edges['id_entity_2'].map(rawid_to_local).to_numpy()
t_type  = df_edges['id_entity_2'].map(rawid_to_type).to_numpy()
rel_idx = df_edges['predicate'].to_numpy()

# Эти тензоры ты будешь отдавать в DataLoader
h_local_tensor = torch.tensor(h_local, dtype=torch.long)
h_type_tensor  = torch.tensor(h_type, dtype=torch.long)
t_local_tensor = torch.tensor(t_local, dtype=torch.long)
t_type_tensor  = torch.tensor(t_type, dtype=torch.long)
rel_tensor     = torch.tensor(rel_idx, dtype=torch.long)

In [6]:
h_local_tensor.min()

tensor(0)

In [5]:
class GraphDataset(Dataset):
    def __init__(self, h_local, h_type, rel, t_local, t_type):
        self.h_local = h_local
        self.h_type = h_type
        self.rel = rel
        self.t_local = t_local
        self.t_type = t_type

    def __len__(self):
        return len(self.h_local)

    def __getitem__(self, idx):
        return (
            self.h_local[idx],
            self.h_type[idx],
            self.rel[idx],
            self.t_local[idx],
            self.t_type[idx]
        )

In [6]:
class MultiModalDistMult(nn.Module):
    def __init__(self, raw_tensors, output_dim, num_relations):
        super().__init__()
        self.output_dim = output_dim
        self.num_types = len(raw_tensors)

        # 1. Хранилища сырых эмбеддингов
        self.raw_stores = nn.ModuleDict({
            str(t_id): nn.Embedding.from_pretrained(tensor, freeze=True)
            for t_id, tensor in raw_tensors.items()
        })

        # 2. Создаем MLP-проекторы автоматически под размер входа каждого тензора
        self.projections = nn.ModuleDict()
        for t_id, tensor in raw_tensors.items():
            in_dim = tensor.shape[1] # Узнаем исходную размерность (например, 1024 для белков)
            self.projections[str(t_id)] = nn.Sequential(
                nn.Linear(in_dim, output_dim * 2),
                nn.ReLU(),
                nn.LayerNorm(output_dim * 2), # Нормализация внутри MLP
                nn.Linear(output_dim * 2, output_dim)
            )

        # 3. Эмбеддинги отношений для DistMult
        self.rel_emb = nn.Embedding(num_relations, output_dim)

    def forward(self, h_local, h_type, r_idx, t_local, t_type):
        batch_size = h_local.size(0)
        device = h_local.device

        # Заготовки под финальные векторы размерности D
        h_proj = torch.zeros(batch_size, self.output_dim, device=device)
        t_proj = torch.zeros(batch_size, self.output_dim, device=device)

        # Перебираем 4 типа данных
        for type_id in range(self.num_types):
            t_str = str(type_id)

            # Находим элементы этого типа в батче
            h_mask = (h_type == type_id)
            t_mask = (t_type == type_id)

            # Проецируем головы
            if h_mask.any():
                raw_h = self.raw_stores[t_str](h_local[h_mask]).float()
                h_proj[h_mask] = self.projections[t_str](raw_h)

            # Проецируем хвосты
            if t_mask.any():
                raw_t = self.raw_stores[t_str](t_local[t_mask]).float()
                t_proj[t_mask] = self.projections[t_str](raw_t)

        # L2-нормализация (обязательно для DistMult!)
        h_proj = F.normalize(h_proj, p=2, dim=1)
        t_proj = F.normalize(t_proj, p=2, dim=1)

        # Достаем эмбеддинги отношений
        r = self.rel_emb(r_idx)

        # Считаем скор DistMult: <h, r, t>
        score = torch.sum(h_proj * r * t_proj, dim=1)
        return score



In [16]:
def prepare_evaluation(model, all_h_local, all_h_type, all_r, all_t_local, all_t_type, device):
    model.eval()

    # 1. Проецируем ВСЕ узлы графа в общее пространство D
    all_embs = []
    entity2eval_id = {} # Маппинг (local_id, type_id) -> индекс в матрице оценки
    eval_id = 0

    with torch.no_grad():
        for type_id in range(model.num_types):
            type_str = str(type_id)
            num_nodes = model.raw_stores[type_str].weight.size(0)

            # Достаем и проецируем все узлы этого типа
            locs = torch.arange(num_nodes, device=device)
            raw = model.raw_stores[type_str](locs).float()
            proj = model.projections[type_str](raw)
            proj = F.normalize(proj, p=2, dim=1)

            all_embs.append(proj)

            # Запоминаем, какая строка новой матрицы какому узлу принадлежит
            for local_id in range(num_nodes):
                entity2eval_id[(local_id, type_id)] = eval_id
                eval_id += 1

    # Матрица всех узлов графа [N_total, D]
    E_all = torch.cat(all_embs, dim=0)

    # 2. Создаем словарь для фильтрации (h_eval_id, r) -> set(t_eval_ids)
    filter_dict = defaultdict(set)

    # Проходимся по ВСЕМ ребрам графа
    for i in range(len(all_h_local)):
        h_key = (all_h_local[i].item(), all_h_type[i].item())
        t_key = (all_t_local[i].item(), all_t_type[i].item())
        r = all_r[i].item()

        h_eval_id = entity2eval_id[h_key]
        t_eval_id = entity2eval_id[t_key]

        filter_dict[(h_eval_id, r)].add(t_eval_id)

    return E_all, entity2eval_id, filter_dict



def evaluate_filtered(model, val_dataloader, E_all, entity_to_eval_id, filter_dict, device, k_list=[1, 5, 10, 50]):
    model.eval()

    mrr = 0
    hits = {k: 0 for k in k_list}
    total_samples = 0

    with torch.no_grad():
        for i, batch in enumerate(val_dataloader):
            if i >= 50:
                break

            h_loc, h_typ, r_idx, t_loc, t_typ = [b.to(device) for b in batch]
            batch_size = h_loc.size(0)
            total_samples += batch_size

            # 1. Получаем векторы голов и отношений
            h_eval_ids = [entity_to_eval_id[(local.item(), typ.item())] for local, typ in zip(h_loc, h_typ)]

            h_proj = E_all[torch.tensor(h_eval_ids, device=device)]
            r_emb = model.rel_emb(r_idx)

            # 2. Считаем скоры
            query_emb = h_proj * r_emb
            all_scores = torch.matmul(query_emb, E_all.T) # [batch_size, N_total]

            # 3. Пакетная фильтрация
            mask_b = []
            mask_idx = []
            target_scores = torch.zeros(batch_size, device=device)

            # Быстро собираем индексы для маскирования на CPU
            for i in range(batch_size):
                h_key = (h_loc[i].item(), h_typ[i].item())
                t_key = (t_loc[i].item(), t_typ[i].item())
                r = r_idx[i].item()

                h_eval_id = entity_to_eval_id[h_key]
                target_t_eval_id = entity_to_eval_id[t_key]

                # Сохраняем целевой скор (чтобы маскирование не сломало его)
                target_scores[i] = all_scores[i, target_t_eval_id]

                true_tails = filter_dict[(h_eval_id, r)]
                for true_t in true_tails:
                    if true_t != target_t_eval_id:
                        mask_b.append(i)
                        mask_idx.append(true_t)

            # Применяем маску ко всем элементам разом (одна операция на GPU)
            if mask_b:
                all_scores[mask_b, mask_idx] = -1e9

            # 4. Векторизованное вычисление рангов для всего батча
            # Сравниваем всю матрицу скоров с вектором целевых скоров
            ranks = (all_scores > target_scores.unsqueeze(1)).sum(dim=1) + 1

            # Аккумулируем метрики
            mrr += (1.0 / ranks).sum().item()
            for k in k_list:
                hits[k] += (ranks <= k).sum().item()

            # Очищаем память от тяжелой матрицы перед следующим батчем
            del all_scores, query_emb


        mrr = mrr / total_samples
        hits = {k: v/total_samples for k, v in hits.items()}

            return mrr, hits


In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = GraphDataset(h_local_tensor, h_type_tensor, rel_tensor, t_local_tensor, t_type_tensor)
train_set, val_set, test_set = random_split(dataset, [0.8, 0.1, 0.1])
train_loader = DataLoader(dataset=train_set, batch_size=2048, shuffle=True)
test_loader = DataLoader(dataset=test_set, batch_size=128, shuffle=True)



# Инициализируем модель
# num_relations = количество уникальных предикатов из len(pred_values)
model = MultiModalDistMult(raw_tensors=raw_tensors, output_dim=128, num_relations=len(pred_values)).to(device)


In [18]:
# Инициализируем оптимизатор и функцию потерь
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# MarginRankingLoss будет заставлять позитивный скор быть больше негативного на величину margin
loss_fn = nn.MarginRankingLoss(margin=1.0)

model.train()

for epoch in range(3): # Количество эпох
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        # 1. Достаем реальные данные (позитивные примеры)
        h_loc, h_typ, r_id, t_loc, t_typ = [b.to(device) for b in batch]
        batch_size = h_loc.size(0)

        # 2. Считаем скор для РЕАЛЬНЫХ триплетов
        pos_scores = model(h_loc, h_typ, r_id, t_loc, t_typ)

        # 3. ГЕНЕРАЦИЯ НЕГАТИВНЫХ ПРИМЕРОВ (In-batch corruption)
        # Создаем случайную перестановку индексов для этого батча
        perm = torch.randperm(batch_size, device=device)

        # Перемешиваем хвосты и их типы (чтобы связка локальный ID + тип не разорвалась)
        neg_t_loc = t_loc[perm]
        neg_t_typ = t_typ[perm]

        # 4. Считаем скор для ФАЛЬШИВЫХ триплетов
        neg_scores = model(h_loc, h_typ, r_id, neg_t_loc, neg_t_typ)

        # 5. Считаем ошибку
        # Таргет = 1 означает, что мы хотим: pos_scores > neg_scores + margin
        target = torch.ones_like(pos_scores)
        loss = loss_fn(pos_scores, neg_scores, target)

        # 6. Обратное распространение и шаг оптимизатора
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

Epoch 1 | Loss: 0.1567
Epoch 2 | Loss: 0.1097
Epoch 3 | Loss: 0.0992


In [19]:
E_all, entity_to_eval_id, filter_dict = prepare_evaluation(
    model, h_local_tensor, h_type_tensor, rel_tensor,
    t_local_tensor, t_type_tensor, device
)

In [20]:
val_mrr, val_hits10 = evaluate_filtered(
    model, test_loader, E_all, entity_to_eval_id, filter_dict, device
)

In [21]:
val_hits10

{1: 0.0234375, 5: 0.125, 10: 0.1484375, 50: 0.2890625}